# Data Preparation (Apache Spark)

## 1. Import Libraries

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import col, count, when, isnan, isnull
import os

# Initialize Spark session
spark = SparkSession.builder \
    .appName("Data Preparation for Model Training") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

# Config

In [ ]:
from config import RAW_PATH, PREPARED_DIR, SPECIES_INDEXER_MODEL_PATH, PREPARED_DATA_SPARK_PATH, PREPARED_DATA_PARQUET_SPARK_PATH

## 2. Load Data

In [ ]:
# Load the dataset
raw_path = RAW_PATH
df = spark.read.csv(raw_path, header=True, inferSchema=True)
print(f"Dataset shape: ({df.count()}, {len(df.columns)})")
df.show()

## 3. Explore Data

### Data Types (df.dtypes)

In [ ]:
df.dtypes

### Data Info (df.info())

In [ ]:
df.printSchema()

### Check for Missing Value

In [ ]:
# Check for missing values
# Note: isnan() only works with numeric types, so we handle string columns with isnull() only
from pyspark.sql.types import NumericType

print("Missing Values:")
missing_exprs = []
for c in df.columns:
    # Check if column is numeric type
    if isinstance(df.schema[c].dataType, NumericType):
        missing_exprs.append(count(when(isnull(col(c)) | isnan(col(c)), c)).alias(c))
    else:
        # For non-numeric types (like strings), only check for null
        missing_exprs.append(count(when(isnull(col(c)), c)).alias(c))

missing_counts = df.select(missing_exprs)
missing_counts.show()

# Calculate total missing
total_missing = 0
for c in df.columns:
    if isinstance(df.schema[c].dataType, NumericType):
        total_missing += df.filter(isnull(col(c)) | isnan(col(c))).count()
    else:
        total_missing += df.filter(isnull(col(c))).count()

print(f"\nTotal missing values: {total_missing}")

### Statistical summary

In [ ]:
df.describe().show()

## 4. Transform Data

### 4.1 Convert columns Data Types

In [ ]:
from pyspark.sql.functions import upper

# Convert to uppercase
df = df.withColumn('species', upper(col('species')))
df.show()

### 4.2 Encoding

In [ ]:
from pyspark.ml.feature import StringIndexer

# Label encoding for 'species' column
indexer = StringIndexer(inputCol="species", outputCol="species_index")
indexer_model = indexer.fit(df)
df = indexer_model.transform(df)

# Drop the original 'species' column if you don't need it
df = df.drop('species')

# Rename the encoded column if desired
df = df.withColumnRenamed('species_index', 'species')

print("Columns after label encoding:")
print(df.columns)
df.show()

In [ ]:
# Print label -> index mapping
print("Label to index mapping:")
for idx, label in enumerate(indexer_model.labels):
    print(f"{idx}: {label}")


### 4.3 Reorder Columns (Result as last column)

In [ ]:
# Nothing to reorder

print("Reordered columns:")
print(df.columns)
df.show()

## 5. Data Summary

### data types

In [ ]:
df.dtypes

### data info

In [ ]:
df.printSchema()

### df (data)

In [ ]:
df.show()

## 6. Save Prepared Data

### prepare output directory

In [ ]:
output_dir = PREPARED_DIR
os.makedirs(output_dir, exist_ok=True)

### Save Indexes

In [ ]:
# Save the fitted StringIndexerModel for future reference
indexer_model_path = SPECIES_INDEXER_MODEL_PATH
indexer_model.write().overwrite().save(indexer_model_path)
print(f"✓ StringIndexerModel saved to {indexer_model_path}")

# # Example: Load the saved model (for inference or reverse mapping)
# loaded_indexer_model = StringIndexerModel.load("../target/prepared/species_indexer_model")
# print("Loaded labels from saved model:", loaded_indexer_model.labels)

### Save as CSV

In [ ]:
csv_path = PREPARED_DATA_SPARK_PATH
# Spark writes to a directory, so we use coalesce(1) to get a single file
df.coalesce(1).write.mode('overwrite').option('header', 'true').csv(csv_path)
print(f"✓ Saved CSV: {csv_path}")

### Save as Parquet

In [ ]:
parquet_path = PREPARED_DATA_PARQUET_SPARK_PATH
df.coalesce(1).write.mode('overwrite').parquet(parquet_path)
print(f"✓ Saved Parquet: {parquet_path}")

### Verify saved files

In [ ]:
print("Saved files/directories:")
for f in os.listdir(output_dir):
    filepath = os.path.join(output_dir, f)
    if os.path.isdir(filepath):
        # For Spark output directories, show contents
        total_size = sum(os.path.getsize(os.path.join(filepath, sf)) for sf in os.listdir(filepath))
        print(f"  - {f}/ ({total_size} bytes)")
    else:
        size = os.path.getsize(filepath)
        print(f"  - {f} ({size} bytes)")